In [ ]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from IPython.display import Markdown, display
import pandas as pd

In [ ]:
DATA_DIR = Path("/data/aveera/results")
OUTPUT_DIR = Path("/data/aveera/results/ci_results")

IROF_FILES = ["irof_full_efficientnetb0_test_2026-10-08.csv","irof_full_resnet50_test_2026-10-08.csv",]
RISE_FILE = "rise_original_full.csv"
MAXSENS_FILE="maxsens_checkpoints/maxsens_per_image_checkpoint.csv"

N_BOOT = 10000
ALPHA = 0.05
SEED = 42

MODELS = ["resnet50","efficientnetb0"]
METHODS = ["GradCAM", "LIME", "SHAP"]

LOWER_IS_BETTER = {
    "irof_auc":True,
    "del_auc":True,
    "ins_auc":False,
    "max_sensitivity":True,
    "ssim_dist":True,
}

KEYS = ["model","method","image_index"]

In [ ]:
def load_long() -> pd.DataFrame:
    irof=pd.concat([pd.read_csv(DATA_DIR/f) for f in IROF_FILES])[KEYS+["irof_auc"]]
    rise=pd.read_csv(DATA_DIR/RISE_FILE)[KEYS+["del_auc","ins_auc"]]
    ms=pd.read_csv(DATA_DIR/MAXSENS_FILE)[KEYS+ ["max_sensitivity","ssim_dist"]]

    wide = irof.merge(rise, on=KEYS, how="outer").merge(ms, on=KEYS, how="outer")
    wide = wide[wide["method"].isin(METHODS+["Random"])]
    long = wide.melt(id_vars=KEYS, value_vars=list(LOWER_IS_BETTER),var_name="metric",value_name="value").dropna(subset=["value"],how="all")

    return long

def boot_ci(x,rand):
    """Mean and percentile bootstrap CI. NaNs are dropped"""
    x=np.asarray(x,dtype=float)
    x=x[np.isfinite(x)]
    n=len(x)
    idx=rand.integers(0,n,size=(N_BOOT,n))
    means= x[idx].mean(axis=1)
    low, high = np.percentile(means, [100 * ALPHA / 2, 100*(1-ALPHA/2)])
    return x.mean(), low, high, n

def paired_test(a,b,rand):
    d=(a-b).dropna()
    mean_d, low, high, n =boot_ci(d.values,rand)
    try:
        p=stats.wilcoxon(d.values).pvalue
    except ValueError:
        p=1.0
    return {"n_pairs":n, "mean_diff":mean_d,"ci_low":low,"ci_high":high,"median_diff":d.median(),"p_raw":p}

def add_holm_and_winner(df, family_col):
    df = df.copy()
    df["p_holm"]=np.nan
    for _,grp in df.groupby(family_col):
        df.loc[grp.index,"p_holm"] = multipletests(grp["p_raw"],method="holm")[1]
    df["ci_excludes_0"]=(df["ci_low"]>0) | (df["ci_high"] <0 )
    df["significant_holm"]=df["p_holm"] < ALPHA
    df["evidence"] = np.where(df["significant_holm"] & df["ci_excludes_0"], "both",np.where(df["significant_holm"] | df["ci_excludes_0"], "mixed","none"))

    def winner(r):
        if not r["significant_holm"]:
            return "no reliable difference"
        a_lower = r["mean_diff"] <0
        return r["A"] if a_lower == LOWER_IS_BETTER[r["metric"]] else r["B"]

    df["better_side"] = df.apply(winner,axis=1)
    return df

In [ ]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
rand = np.random.default_rng(SEED)
long = load_long()
piv = long.pivot(index="image_index",columns=["metric","model","method"],values="value")

rows=[]
for metric in LOWER_IS_BETTER:
    for model in MODELS:
        for method in METHODS+["Random"]:
            if (metric, model, method) not in piv.columns:
                continue
            m, low, high, n = boot_ci(piv[(metric,model,method)].values,rand)
            rows.append(dict(metric=metric,model=model,method=method,n=n,mean=m,ci_low=low,ci_high=high))

ci=pd.DataFrame(rows)
ci.to_csv(OUTPUT_DIR/"ci_means.csv",index=False)

ci["mean [95% CI]"] = ci.apply(lambda r: f"{r['mean']:.4f} [{r['ci_low']:.4f}, {r['ci_high']:.4f}]", axis=1)
ci.pivot(index=["model","method"], columns="metric",values="mean [95% CI]") \
      .to_csv(OUTPUT_DIR / "table_mean_ci.csv")

rows=[]
for metric in LOWER_IS_BETTER:
    for model in MODELS:
        for A,B in combinations(METHODS,2):
            rows.append(dict(metric=metric,model=model,A=A,B=B,
                             **paired_test(piv[(metric,model,A)],piv[(metric,model,B)],rand)))

pm = add_holm_and_winner(pd.DataFrame(rows),family_col="metric")
pm.to_csv(OUTPUT_DIR/"paired_methods.csv",index=False)

rows=[]
for metric in LOWER_IS_BETTER:
    for method in METHODS:
        rows.append(dict(metric=metric,method=method,A="efficientnetb0",B="resnet50",
                         **paired_test(piv[(metric,"efficientnetb0",method)],
                                       piv[(metric,"resnet50",method)],rand)))
pa=add_holm_and_winner(pd.DataFrame(rows),family_col="metric")
pa.to_csv(OUTPUT_DIR/"paired_architectures.csv",index=False)

rows=[]
for metric in ["del_auc","ins_auc"]:
    for model in MODELS:
        for method in METHODS:
            rows.append(dict(metric=metric,model=model,A=method,B="Random",
                             **paired_test(piv[(metric,model,method)],
                                           piv[(metric,model,"Random")],
                                           rand)))
pr= add_holm_and_winner(pd.DataFrame(rows),family_col="metric")
pr.to_csv(OUTPUT_DIR/"vs_random.csv",index=False)

In [ ]:
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)


def show(df: pd.DataFrame) -> pd.DataFrame:
    """Filters, rounds, and resets the index of the comparison DataFrame."""
    optional_cols = [col for col in ("model", "method") if col in df.columns]

    core_cols = [
        "A",
        "B",
        "n_pairs",
        "mean_diff",
        "ci_low",
        "ci_high",
        "p_holm",
        "evidence",
        "better_side",
    ]

    keep_columns = ["metric"] + optional_cols + core_cols
    return df[keep_columns].round(4).reset_index(drop=True)


display(Markdown("### Mean [95% bootstrap CI]"))
pivot_df = ci.pivot(
    index=["model", "method"], columns="metric", values="mean [95% CI]"
)
display(pivot_df)

display(Markdown("### Method vs method (diff = A − B, paired by image)"))
display(show(pm))

display(
    Markdown(
        "### Architecture (diff = EfficientNetB0 − ResNet50, paired by image)"
    )
)
display(show(pa))

display(Markdown("### Method vs Random baseline (diff = method − Random)"))
display(show(pr))

print(f"Saved CSVs to {OUTPUT_DIR}")
